# Bone Suppression Retraining v1

Run this notebook in Kaggle with GPU enabled. It uses the public `hmchuong/xray-bone-shadow-supression` dataset, creates deterministic train/validation/test splits, trains both supported models, evaluates autoregressive steps with `device=auto`, and writes checkpoints, metrics, manifests, and comparison panels under `/kaggle/working/training_runs/retrained_v1`.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
assert (REPO_ROOT / 'pyproject.toml').exists(), 'Run this notebook from the repository root.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,gan,unet]'], check=True)

In [ ]:
DATASET_ROOT = Path('/kaggle/input/xray-bone-shadow-supression')
RUN_DIR = Path('/kaggle/working/training_runs/retrained_v1')
SPLITS = RUN_DIR / 'splits.json'
RUN_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT, RUN_DIR

In [ ]:
!bone-suppression-repro prepare-splits --dataset-root {DATASET_ROOT} --output {SPLITS} --seed 2026

In [ ]:
!bone-suppression-repro train --model gan_mso2 --dataset-root {DATASET_ROOT} --splits {SPLITS} --output-dir {RUN_DIR / 'gan_mso2'} --epochs 50 --batch-size 4 --image-size 256 --learning-rate 0.0002 --seed 2026

In [ ]:
!bone-suppression-repro train --model unet_resnet50 --dataset-root {DATASET_ROOT} --splits {SPLITS} --output-dir {RUN_DIR / 'unet_resnet50'} --epochs 50 --batch-size 4 --image-size 256 --learning-rate 0.0002 --seed 2026

In [ ]:
!bone-suppression-repro evaluate-steps --model gan_mso2 --checkpoint {RUN_DIR / 'gan_mso2' / 'gan_mso2_retrained_v1.keras'} --dataset-root {DATASET_ROOT} --splits {SPLITS} --output-dir {RUN_DIR / 'evaluation'} --device auto --split test --steps 0,1,2,3,4,5
!bone-suppression-repro evaluate-steps --model unet_resnet50 --checkpoint {RUN_DIR / 'unet_resnet50' / 'unet_resnet50_retrained_v1.pkl'} --dataset-root {DATASET_ROOT} --splits {SPLITS} --output-dir {RUN_DIR / 'evaluation'} --device auto --split test --steps 0,1,2,3,4,5


In [ ]:
!bone-suppression-repro merge-metrics --output {RUN_DIR / 'metrics.json'} {RUN_DIR / 'evaluation' / 'gan_mso2_test_metrics.json'} {RUN_DIR / 'evaluation' / 'unet_resnet50_test_metrics.json'}
!bone-suppression-repro comparison-examples --dataset-root {DATASET_ROOT} --splits {SPLITS} --output-dir {RUN_DIR / 'examples'} --gan-predictions {RUN_DIR / 'evaluation' / 'predictions' / 'gan_mso2'} --unet-predictions {RUN_DIR / 'evaluation' / 'predictions' / 'unet_resnet50'} --count 3

In [ ]:
print('Artifacts ready in:', RUN_DIR)
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(RUN_DIR))